In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
from run.preprocess.config import _project_path

# ============================================================
# CONFIGURAZIONE
# ============================================================

ROOT = Path(
    _project_path(
        "experiments/cnn1D/raw_segment(train_seg - val_no)"
    )
)

# Metriche disponibili nei risultati
METRIC = "val_accuracy"

print("Root:", ROOT)
print("Exists:", ROOT.exists())

Root: C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)
Exists: True


In [3]:
def find_result_csvs(root: Path):
    """
    Cerca ricorsivamente tutti i file CSV
    all'interno della directory degli esperimenti.
    """

    csv_files = list(root.rglob("*.csv"))

    print(f"CSV trovati: {len(csv_files)}")

    for path in csv_files[:20]:
        print(path)

    if len(csv_files) > 20:
        print("...")

    return csv_files


csv_files = find_result_csvs(ROOT)

CSV trovati: 1476
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_1_best_metrics.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_1_history.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_2_best_metrics.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_2_history.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_3_best_metrics.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_3_history.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Task_all\acceleration\fold_4_best_metrics.csv
C:\Users\Utente\parkinson_disease\experiments\cnn1D\raw_segment(train_seg - val_no)\Tas

In [4]:
def find_accuracy_column(df: pd.DataFrame):
    """
    Cerca la colonna contenente la validation accuracy.
    """

    candidates = [
        "val_accuracy",
        "validation_accuracy",
        "accuracy",
    ]

    for column in candidates:
        if column in df.columns:
            return column

    return None

In [5]:
records = []

for csv_path in csv_files:

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Errore lettura {csv_path}: {e}")
        continue

    accuracy_column = find_accuracy_column(df)

    if accuracy_column is None:
        continue

    # --------------------------------------------------------
    # Trova la cartella Task_*
    # --------------------------------------------------------

    task_name = None

    for parent in csv_path.parents:

        match = re.fullmatch(
            r"Task_(.+)",
            parent.name
        )

        if match:
            task_name = match.group(1)
            break

        # evita di risalire oltre la root
        if parent == ROOT:
            break

    if task_name is None:
        continue

    # --------------------------------------------------------
    # Identifica la feature/configurazione
    # --------------------------------------------------------

    relative_path = csv_path.relative_to(ROOT)

    parts = relative_path.parts

    # esempio:
    # Task_1 / acceleration / results.csv
    #
    # oppure:
    # Task_1 / xy_speed_velocity... / results.csv

    task_index = None

    for i, part in enumerate(parts):
        if re.fullmatch(r"Task_.+", part):
            task_index = i
            break

    if task_index is None:
        continue

    feature_parts = parts[
        task_index + 1:-1
    ]

    if feature_parts:
        feature_name = "/".join(feature_parts)
    else:
        feature_name = "unknown"

    # --------------------------------------------------------
    # Prende il valore migliore presente nel CSV
    # --------------------------------------------------------

    values = pd.to_numeric(
        df[accuracy_column],
        errors="coerce"
    ).dropna()

    if values.empty:
        continue

    best_accuracy = values.max()

    records.append({

        "task": f"Task_{task_name}",

        "feature": feature_name,

        "accuracy": best_accuracy,

        "source_file": str(
            csv_path
        ),
    })


results_df = pd.DataFrame(records)

print(
    f"Risultati validi trovati: "
    f"{len(results_df)}"
)

results_df.head()

Risultati validi trovati: 1476


,task,feature,accuracy,source_file
0,Task_all,acceleration,0.533333,C:\Users\Utente\parkinson_disease\experiments\...
1,Task_all,acceleration,0.533333,C:\Users\Utente\parkinson_disease\experiments\...
2,Task_all,acceleration,0.533333,C:\Users\Utente\parkinson_disease\experiments\...
3,Task_all,acceleration,0.533333,C:\Users\Utente\parkinson_disease\experiments\...
4,Task_all,acceleration,0.466667,C:\Users\Utente\parkinson_disease\experiments\...


In [15]:
import pandas as pd
from pathlib import Path

ROOT = Path(
    _project_path(
        "experiments/cnn1D/raw_segment(train_seg - val_no)"
    )
)

results = []

# ============================================================
# CERCA SOLO I SUMMARY.CSV
# ============================================================

summary_files = list(ROOT.rglob("summary.csv"))

print(f"Summary trovati: {len(summary_files)}")

for csv_path in summary_files:

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Errore: {csv_path} -> {e}")
        continue

    if df.empty:
        continue

    if "val_accuracy" not in df.columns:
        continue

    # summary.csv contiene una sola riga aggregata
    row = df.iloc[0]

    # --------------------------------------------------------
    # Trova la cartella Task_X
    # --------------------------------------------------------

    task_name = None
    task_index = None

    parts = csv_path.relative_to(ROOT).parts

    for i, part in enumerate(parts):

        if part.startswith("Task_"):
            task_name = part
            task_index = i
            break

    if task_name is None:
        continue

    # --------------------------------------------------------
    # Nome della configurazione feature
    # --------------------------------------------------------

    if task_index + 1 < len(parts) - 1:
        feature = parts[task_index + 1]
    else:
        feature = "unknown"

    # --------------------------------------------------------
    # Risultato aggregato
    # --------------------------------------------------------

    results.append({
        "task": task_name,
        "feature": feature,
        "val_accuracy": float(row["val_accuracy"]),
        "val_f1": float(row["val_f1"]),
        "val_auc": float(row["val_auc"]),
        "val_precision": float(row["val_precision"]),
        "val_recall": float(row["val_recall"]),
        "val_specificity": float(row["val_specificity"]),
        "val_loss": float(row["val_loss"]),
        "source": str(csv_path),
    })


results_df = pd.DataFrame(results)

print(f"Esperimenti caricati: {len(results_df)}")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(results_df)

Summary trovati: 123
Esperimenti caricati: 123


,task,feature,val_accuracy,val_f1,val_auc,val_precision,val_recall,val_specificity,val_loss,source
0,Task_all,acceleration,0.493333,0.660079,0.651786,0.493333,1.000000,0.000000,0.695147,C:\Users\Utente\parkinson_disease\experiments\...
1,Task_all,altitude,0.546667,0.382057,0.667857,0.488788,0.457143,0.639286,0.714164,C:\Users\Utente\parkinson_disease\experiments\...
2,Task_all,azimuth,0.586667,0.627937,0.646429,0.559805,0.725000,0.442857,0.688077,C:\Users\Utente\parkinson_disease\experiments\...
3,Task_all,button,0.493333,0.660079,0.701786,0.493333,1.000000,0.000000,0.697216,C:\Users\Utente\parkinson_disease\experiments\...
4,Task_all,pressure,0.533333,0.615355,0.692857,0.499580,0.828571,0.225000,0.692992,C:\Users\Utente\parkinson_disease\experiments\...
5,Task_all,pressure_derivative,0.640000,0.612506,0.771429,0.696190,0.692857,0.571429,0.662293,C:\Users\Utente\parkinson_disease\experiments\...
6,Task_all,speed,0.680000,0.687980,0.800000,0.665000,0.750000,0.596429,0.664355,C:\Users\Utente\parkinson_disease\experiments\...
7,Task_all,timestamp,0.493333,0.660079,0.641071,0.493333,1.000000,0.000000,0.698274,C:\Users\Utente\parkinson_disease\experiments\...
8,Task_all,velocity,0.600000,0.425000,0.775000,0.581111,0.407143,0.792857,0.679645,C:\Users\Utente\parkinson_disease\experiments\...
9,Task_all,xy_acceleration_velocity_pressure_derivative,0.760000,0.656899,0.921429,0.966667,0.532143,0.975000,0.642813,C:\Users\Utente\parkinson_disease\experiments\...
